In [20]:
from Bio import SeqIO

records = list(
    SeqIO.parse(
        "../data/raw/BindingDBTargetSequences.fasta",
        "fasta"
    )
)

print(
    len(records)
)

11433


In [42]:
print(records[0].id)
print(records[0].description)

p1
p1 mol:protein length:376 Thymidine kinase


In [43]:
print(records[100].id)
print(records[100].description)

p151
p151 mol:protein length:99 HIV-1 Protease Mutant (Q7K/L33I/L63I) chain A


In [44]:
protein_df = pd.DataFrame({
    "Protein_ID": [r.id for r in records],
    "Description": [r.description for r in records],
    "Sequence": [str(r.seq) for r in records]
})

protein_df.head()

,Protein_ID,Description,Sequence
0,p1,p1 mol:protein length:376 Thymidine kinase,MASYPCHQHASAFDQAARSRGHNNRRTALRPRRQQKATEVRLEQKM...
1,p2,p2 mol:protein length:159 Streptavadin(N23A),DPSKDSKAQVSAAEAGITGTWYAQLGSTFIVTAGADGALTGTYESA...
2,p3,p3 mol:protein length:159 Streptavadin(Y43A),DPSKDSKAQVSAAEAGITGTWYNQLGSTFIVTAGADGALTGTAESA...
3,p4,p4 mol:protein length:159 Streptavadin(S27A),DPSKDSKAQVSAAEAGITGTWYNQLGATFIVTAGADGALTGTYESA...
4,p5,p5 mol:protein length:183 Streptavidin,MRKIVVAAIAVSLTTVSITASASADPSKDSKAQVSAAEAGITGTWY...


In [21]:
records[0]

SeqRecord(seq=Seq('MASYPCHQHASAFDQAARSRGHNNRRTALRPRRQQKATEVRLEQKMPTLLRVYI...EAN'), id='p1', name='p1', description='p1 mol:protein length:376 Thymidine kinase', dbxrefs=[])

create data frame

Sequence Length Distribution

In [23]:
protein_df["Length"] = (
    protein_df["Sequence"]
    .apply(len)
)

protein_df["Length"].describe()

count    11433.000000
mean       595.359311
std        653.545961
min         51.000000
25%        310.000000
50%        451.000000
75%        690.000000
max      34350.000000
Name: Length, dtype: float64

In [45]:
protein_df.to_csv(
    "../data/processed/protein_sequences.csv",
    index=False
)

In [24]:
def encode_protein(
    seq
):

    return [
        AA_VOCAB[a]
        for a in seq
        if a in AA_VOCAB
    ]

In [25]:
AA_VOCAB = {
    "A":0,
    "C":1,
    "D":2,
    "E":3,
    "F":4,
    "G":5,
    "H":6,
    "I":7,
    "K":8,
    "L":9,
    "M":10,
    "N":11,
    "P":12,
    "Q":13,
    "R":14,
    "S":15,
    "T":16,
    "V":17,
    "W":18,
    "Y":19
}

In [26]:
encode_protein(
    protein_df.iloc[0]["Sequence"][:20]
)

[10, 0, 15, 19, 12, 1, 6, 13, 6, 0, 15, 0, 4, 2, 13, 0, 0, 14, 15, 14]

In [27]:
protein_df["Length"] = (
    protein_df["Sequence"]
    .apply(len)
)

protein_df["Length"].describe()

count    11433.000000
mean       595.359311
std        653.545961
min         51.000000
25%        310.000000
50%        451.000000
75%        690.000000
max      34350.000000
Name: Length, dtype: float64

In [28]:
protein_df["Tokens"] = (
    protein_df["Sequence"]
    .apply(encode_protein)
)

protein_df.head()

,Protein_ID,Sequence,Length,Tokens
0,p1,MASYPCHQHASAFDQAARSRGHNNRRTALRPRRQQKATEVRLEQKM...,376,"[10, 0, 15, 19, 12, 1, 6, 13, 6, 0, 15, 0, 4, ..."
1,p2,DPSKDSKAQVSAAEAGITGTWYAQLGSTFIVTAGADGALTGTYESA...,159,"[2, 12, 15, 8, 2, 15, 8, 0, 13, 17, 15, 0, 0, ..."
2,p3,DPSKDSKAQVSAAEAGITGTWYNQLGSTFIVTAGADGALTGTAESA...,159,"[2, 12, 15, 8, 2, 15, 8, 0, 13, 17, 15, 0, 0, ..."
3,p4,DPSKDSKAQVSAAEAGITGTWYNQLGATFIVTAGADGALTGTYESA...,159,"[2, 12, 15, 8, 2, 15, 8, 0, 13, 17, 15, 0, 0, ..."
4,p5,MRKIVVAAIAVSLTTVSITASASADPSKDSKAQVSAAEAGITGTWY...,183,"[10, 14, 8, 7, 17, 17, 0, 0, 7, 0, 17, 15, 9, ..."


In [29]:
len(
    protein_df.iloc[0]["Tokens"]
)

376

In [30]:
protein_df.iloc[0]["Tokens"][:20]

[10, 0, 15, 19, 12, 1, 6, 13, 6, 0, 15, 0, 4, 2, 13, 0, 0, 14, 15, 14]

In [31]:
MAX_LEN = 512

def pad_sequence(tokens):

    tokens = tokens[:MAX_LEN]

    padding = [0] * (
        MAX_LEN - len(tokens)
    )

    return tokens + padding

protein_df["InputIDs"] = (
    protein_df["Tokens"]
    .apply(pad_sequence)
)

protein_df.head()

,Protein_ID,Sequence,Length,Tokens,InputIDs
0,p1,MASYPCHQHASAFDQAARSRGHNNRRTALRPRRQQKATEVRLEQKM...,376,"[10, 0, 15, 19, 12, 1, 6, 13, 6, 0, 15, 0, 4, ...","[10, 0, 15, 19, 12, 1, 6, 13, 6, 0, 15, 0, 4, ..."
1,p2,DPSKDSKAQVSAAEAGITGTWYAQLGSTFIVTAGADGALTGTYESA...,159,"[2, 12, 15, 8, 2, 15, 8, 0, 13, 17, 15, 0, 0, ...","[2, 12, 15, 8, 2, 15, 8, 0, 13, 17, 15, 0, 0, ..."
2,p3,DPSKDSKAQVSAAEAGITGTWYNQLGSTFIVTAGADGALTGTAESA...,159,"[2, 12, 15, 8, 2, 15, 8, 0, 13, 17, 15, 0, 0, ...","[2, 12, 15, 8, 2, 15, 8, 0, 13, 17, 15, 0, 0, ..."
3,p4,DPSKDSKAQVSAAEAGITGTWYNQLGATFIVTAGADGALTGTYESA...,159,"[2, 12, 15, 8, 2, 15, 8, 0, 13, 17, 15, 0, 0, ...","[2, 12, 15, 8, 2, 15, 8, 0, 13, 17, 15, 0, 0, ..."
4,p5,MRKIVVAAIAVSLTTVSITASASADPSKDSKAQVSAAEAGITGTWY...,183,"[10, 14, 8, 7, 17, 17, 0, 0, 7, 0, 17, 15, 9, ...","[10, 14, 8, 7, 17, 17, 0, 0, 7, 0, 17, 15, 9, ..."


In [32]:
len(
    protein_df.iloc[0]["InputIDs"]
)

512

In [33]:
import torch
import torch.nn as nn

In [34]:
class ProteinEncoder(nn.Module):

    def __init__(self):

        super().__init__()

        self.embedding = nn.Embedding(
            20,
            128
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=128,
            nhead=8,
            batch_first=True
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=2
        )

    def forward(
        self,
        x
    ):

        x = self.embedding(x)

        x = self.transformer(x)

        return x.mean(
            dim=1
        )

In [35]:
model = ProteinEncoder()

sample = torch.tensor(
    [
        protein_df.iloc[0]["InputIDs"]
    ]
)

output = model(sample)

print(
    output.shape
)

torch.Size([1, 128])


In [41]:
protein_df.to_csv(
    "../data/processed/protein_sequences.csv",
    index=False
)

In [46]:
protein_df["Target_Name"] = (
    protein_df["Description"]
    .str.replace(
        r"^p\d+\s+mol:protein\s+length:\d+\s*",
        "",
        regex=True
    )
)

protein_df[
    ["Protein_ID", "Target_Name"]
].head()

,Protein_ID,Target_Name
0,p1,Thymidine kinase
1,p2,Streptavadin(N23A)
2,p3,Streptavadin(Y43A)
3,p4,Streptavadin(S27A)
4,p5,Streptavidin


In [47]:
protein_df["Target_Name"].sample(20)

4476                                 Endothelin-1 receptor
9152                                         Neuraminidase
10259                        ATP-dependent DNA helicase Q1
5964     Dual specificity mitogen-activated protein kin...
3971                               Melanocortin receptor 5
7318                                        Protein FAM83A
2403                     Alkanal monooxygenase alpha chain
6193           NLR family CARD domain-containing protein 4
2945                    Mitogen-activated protein kinase 3
1268                Phenylethanolamine N-methyltransferase
6938                                            E-selectin
5586                                    Beta-glucuronidase
5937                       cGMP-dependent protein kinase 2
250                   Gag-Pol polyprotein [588-1127,L687I]
9464     Pleiotropic ABC efflux transporter of multiple...
5609           Sodium channel protein type 5 subunit alpha
1564               Interleukin-17 receptor A [N206D,N265

In [49]:
import pandas as pd

binding_df = pd.read_csv(
    "../data/processed/bindingdb_clean.csv"
)

print(binding_df.shape)
binding_df.head()

(21525, 4)


,Ligand SMILES,Target Name,Ki (nM),pKi
0,CCC(c1ccccc1)c1c(O)c2ccccc2oc1=O,Dimer of Gag-Pol polyprotein [489-587],1000.0,6.000000
1,CCC(c1ccccc1)c1c(O)cc(CCc2ccccc2)oc1=O,Dimer of Gag-Pol polyprotein [489-587],500.0,6.301030
2,CCC(Cc1ccccc1)c1cc(O)c(C(CC)c2ccccc2)c(=O)o1,Dimer of Gag-Pol polyprotein [489-587],38.0,7.420216
3,Oc1c2CCCCCCc2oc(=O)c1C(C1CC1)c1ccccc1,Dimer of Gag-Pol polyprotein [489-587],15.0,7.823909
4,CCC(Cc1ccccc1)c1cc(O)c(C(CC)c2ccccc2)c(=O)o1,Dimer of Gag-Pol polyprotein [514-612],32.0,7.494850


In [50]:
binding_df["Target Name"].sample(20)

9357                                           Cathepsin S
20133                                 Coagulation factor X
8171                           Methionine aminopeptidase 2
13835                                 Carbonic anhydrase 4
19300                    Fatty acid-binding protein, liver
14397    Dihydroorotate dehydrogenase (quinone), mitoch...
15273                Serine/threonine-protein kinase pim-2
8080                                  Coagulation factor X
1729                            Carboxylic ester hydrolase
16595                               Cannabinoid receptor 1
6353                                     Serine protease 1
10384                      5-hydroxytryptamine receptor 2A
12841                       Carbonic anhydrase 15 [19-324]
14633                       5-hydroxytryptamine receptor 7
12614                                 Carbonic anhydrase 6
5679     Dimer of Gag-Pol polyprotein [501-599,Q508K,L5...
17569      Neuronal acetylcholine receptor subunit alpha

In [51]:
protein_targets = set(
    protein_df["Target_Name"]
)

binding_targets = set(
    binding_df["Target Name"]
)

matches = protein_targets.intersection(
    binding_targets
)

print("Protein Targets:", len(protein_targets))
print("Binding Targets:", len(binding_targets))
print("Exact Matches:", len(matches))

Protein Targets: 8257
Binding Targets: 771
Exact Matches: 647


In [ ]:
merged_df = binding_df.merge(
    protein_df,
    left_on="Target Name",
    right_on="Target_Name",
    how="inner"
)

print(merged_df.shape)

(94610, 8)


In [53]:
merged_df[
    [
        "Ligand SMILES",
        "Target Name",
        "pKi"
    ]
].head()

,Ligand SMILES,Target Name,pKi
0,CN(C)CCCN1c2ccccc2Sc2ccc(Cl)cc12,5-hydroxytryptamine receptor 7,6.318759
1,CN(C)CCCN1c2ccccc2Sc2ccc(Cl)cc12,5-hydroxytryptamine receptor 7,6.318759
2,CN(C)CCCN1c2ccccc2Sc2ccc(Cl)cc12,5-hydroxytryptamine receptor 7,6.318759
3,CN(C)CCCN1c2ccccc2Sc2ccc(Cl)cc12,5-hydroxytryptamine receptor 7,6.318759
4,CN(C)CCCN1c2ccccc2Sc2ccc(Cl)cc12,5-hydroxytryptamine receptor 7,6.318759


In [54]:
merged_df.columns

Index(['Ligand SMILES', 'Target Name', 'Ki (nM)', 'pKi', 'Protein_ID',
       'Description', 'Sequence', 'Target_Name'],
      dtype='str')

In [55]:
merged_df.to_csv(
    "../data/processed/drug_protein_dataset.csv",
    index=False
)

In [56]:
merged_df[
    [
        "Ligand SMILES",
        "Target Name",
        "pKi"
    ]
].head()

,Ligand SMILES,Target Name,pKi
0,CN(C)CCCN1c2ccccc2Sc2ccc(Cl)cc12,5-hydroxytryptamine receptor 7,6.318759
1,CN(C)CCCN1c2ccccc2Sc2ccc(Cl)cc12,5-hydroxytryptamine receptor 7,6.318759
2,CN(C)CCCN1c2ccccc2Sc2ccc(Cl)cc12,5-hydroxytryptamine receptor 7,6.318759
3,CN(C)CCCN1c2ccccc2Sc2ccc(Cl)cc12,5-hydroxytryptamine receptor 7,6.318759
4,CN(C)CCCN1c2ccccc2Sc2ccc(Cl)cc12,5-hydroxytryptamine receptor 7,6.318759


In [59]:
merged_df["pKi"].describe()



count    94610.000000
mean         6.507805
std          1.607843
min         -0.000000
25%          5.481486
50%          6.546162
75%          7.602060
max         12.698970
Name: pKi, dtype: float64

In [58]:
merged_df["Target Name"].nunique()

647